In [1]:
import arcpy
import os
import openpyxl

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================

# Path to your Excel domain definition file
excel_file = r"C:\National Water Utility\Domains.xlsx"

# Target geodatabase
gdb = r"C:\National Water Utility\test.gdb"

# Excel worksheet
worksheet_name = "incidences_domains"

In [3]:


# ============================================================
# CHECK INPUTS
# ============================================================

if not os.path.exists(excel_file):
    raise FileNotFoundError(
        f"Excel file not found: {excel_file}"
    )

if not arcpy.Exists(gdb):
    raise FileNotFoundError(
        f"Geodatabase not found: {gdb}"
    )


# ============================================================
# OPEN EXCEL
# ============================================================

print("Opening Excel workbook...")

workbook = openpyxl.load_workbook(
    excel_file,
    data_only=True
)

if worksheet_name not in workbook.sheetnames:
    raise ValueError(
        f"Worksheet '{worksheet_name}' does not exist."
    )

worksheet = workbook[worksheet_name]


# ============================================================
# READ HEADER ROW
# ============================================================

headers = []

for cell in worksheet[1]:
    headers.append(str(cell.value).strip() if cell.value else "")

required_headers = [
    "DomainName",
    "DomainDescription",
    "FieldType",
    "Code",
    "Description"
]

for header in required_headers:
    if header not in headers:
        raise ValueError(
            f"Missing required Excel column: {header}"
        )

header_index = {
    header: headers.index(header)
    for header in required_headers
}


# ============================================================
# READ DOMAIN DATA
# ============================================================

domains = {}

for row in worksheet.iter_rows(min_row=2, values_only=True):

    domain_name = row[header_index["DomainName"]]
    domain_description = row[header_index["DomainDescription"]]
    field_type = row[header_index["FieldType"]]
    code = row[header_index["Code"]]
    description = row[header_index["Description"]]

    # Skip empty rows
    if not domain_name:
        continue

    domain_name = str(domain_name).strip()
    domain_description = (
        str(domain_description).strip()
        if domain_description
        else domain_name
    )

    field_type = (
        str(field_type).strip().upper()
        if field_type
        else "TEXT"
    )

    if code is None:
        continue

    # Store domain information
    if domain_name not in domains:

        domains[domain_name] = {
            "description": domain_description,
            "field_type": field_type,
            "values": []
        }

    domains[domain_name]["values"].append(
        (
            code,
            description if description else str(code)
        )
    )


# ============================================================
# CREATE DOMAINS
# ============================================================

print("\nCreating domains...\n")

for domain_name, domain_info in domains.items():

    description = domain_info["description"]
    field_type = domain_info["field_type"]

    print(f"Processing domain: {domain_name}")

    # --------------------------------------------------------
    # Check if domain already exists
    # --------------------------------------------------------

    existing_domains = [
        d.name
        for d in arcpy.da.ListDomains(gdb)
    ]

    if domain_name in existing_domains:

        print(
            f"  Domain already exists: {domain_name}"
        )

        continue

    # --------------------------------------------------------
    # Create coded value domain
    # --------------------------------------------------------

    arcpy.management.CreateDomain(
        in_workspace=gdb,
        domain_name=domain_name,
        domain_description=description,
        field_type=field_type,
        domain_type="CODED",
        split_policy="DEFAULT",
        merge_policy="DEFAULT"
    )

    print(
        f"  Created domain: {domain_name}"
    )

    # --------------------------------------------------------
    # Add coded values
    # --------------------------------------------------------

    for code, code_description in domain_info["values"]:

        arcpy.management.AddCodedValueToDomain(
            in_workspace=gdb,
            domain_name=domain_name,
            code=code,
            code_description=str(code_description)
        )

        print(
            f"    Added: {code} = {code_description}"
        )


# ============================================================
# COMPLETED
# ============================================================

print("\n==============================================")
print("DOMAIN CREATION COMPLETED")
print("==============================================")
print(f"Geodatabase: {gdb}")
print(f"Excel file: {excel_file}")
print(f"Domains processed: {len(domains)}")
print("==============================================")

Opening Excel workbook...

Creating domains...

Processing domain: TMaterial
  Domain already exists: TMaterial
Processing domain: PropertyNo
  Domain already exists: PropertyNo
Processing domain: Adaptors
  Domain already exists: Adaptors
Processing domain: Air Valves
  Domain already exists: Air Valves
Processing domain: Bend
  Domain already exists: Bend
Processing domain: Bolts & Nuts
  Domain already exists: Bolts & Nuts
Processing domain: Couplings
  Domain already exists: Couplings
Processing domain: Cover
  Domain already exists: Cover
Processing domain: Elbow
  Domain already exists: Elbow
Processing domain: End Cap
  Domain already exists: End Cap
Processing domain: Flange
  Domain already exists: Flange
Processing domain: Gasket
  Domain already exists: Gasket
Processing domain: Hydrant
  Domain already exists: Hydrant
Processing domain: Nipple
  Domain already exists: Nipple
Processing domain: Pipes_SIZES
  Domain already exists: Pipes_SIZES
Processing domain: Plugs
  Domai